# Train A Self-Driving Lab Policy on H100

This notebook trains a GRPO policy for the **same bio-experiment planning task** as `run_agent.py`: choosing structured actions (collect_sample, run_qc, cluster, de_analysis, etc.) step-by-step in the OpenEnv bio-experiment environment.

**Flow:** Build prompts from `BioExperimentEnvironment` rollouts (same env `run_agent.py` uses) → OpenEnv reward scores actions locally → GRPO trains the model. Uses `build_openenv_reward`, `prepare_prompt_examples`, and `build_grpo_trainer` from `training_script.py`.

In [ ]:
%pip install -q -U torch transformers datasets trl accelerate matplotlib huggingface_hub

# Optional extras used by some reward-scoring paths.
%pip install -q -U sentence-transformers gseapy

In [ ]:
from pathlib import Path

import torch

from training_script import make_training_args, prepare_prompt_examples, run_training

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

Path("artifacts").mkdir(exist_ok=True)

In [ ]:
args = make_training_args(
    model_id="Qwen/Qwen3.5-9B",
    output_dir="artifacts/grpo-h100",
    dataset_episodes=64,                 # more data per run
    rollout_steps=12,                    # slightly longer trajectories
    collection_policy="heuristic",
    reward_backend="local",
    domain_randomise=True,

    num_generations=8,                   # H100 can handle a larger GRPO group
    max_completion_length=192,           # small bump if completions are being cut off
    max_prompt_length=1024,              # trim a bit unless you truly need 1280

    per_device_train_batch_size=8,       # first thing to try on H100
    gradient_accumulation_steps=2,       # same effective batch as before, fewer sync steps
    learning_rate=1e-5,                  # slightly more aggressive for LoRA/QLoRA-style RL tuning
    num_train_epochs=1.0,

    logging_steps=1,
    save_steps=25,
    trust_remote_code=True,
    dry_run=False,
    seed=42,
)

args

In [ ]:
# Same prompt format run_agent.py sees: SYSTEM_PROMPT + observation
preview_data = prepare_prompt_examples(
    make_training_args(
        dataset_episodes=1,
        rollout_steps=args.rollout_steps,
        collection_policy=args.collection_policy,
        scenario_name=["cardiac_disease_de"],
        seed=args.seed,
        domain_randomise=args.domain_randomise,
    )
)
preview_examples = preview_data["examples"]

print(preview_examples[0]["prompt"][:3500])
print("\nReference action:\n", preview_examples[0]["reference_action"])


In [ ]:
# Optional smoke test before a full run.
dry_run_args = make_training_args(**{**vars(args), "dry_run": True})
dry_run_result = run_training(dry_run_args)
len(dry_run_result["examples"])

In [ ]:
from IPython.display import Image, display

train_result = run_training(args)
for name, plot_path in train_result["plot_paths"].items():
    print(name, plot_path)
    display(Image(filename=plot_path))